In [0]:
import logging as l
from pyspark.sql import functions as F
from pyspark.sql import SparkSession 

In [0]:
bronze_df = spark.sql("SELECT * FROM muse.bronze.job_muse_bronze_landing")

silver_df = bronze_df.select(
    F.col("id"),
    F.col("name"),
    F.col("company.id").alias("company_id"),
    F.col("company.name").alias("company_name"),
    F.try_element_at("locations", F.lit(1))["name"].alias("location"),
    F.try_element_at("levels", F.lit(1))["name"].alias("level"),
    F.col("type"),
    F.col("model_type"),
    F.to_timestamp("publication_date").alias("publication_date"),
    F.col("refs.landing_page").alias("job_url"),
    F.col("short_name"),
    F.col("_ingestion_date")
).dropDuplicates(["id"])

silver_df.show(5, truncate=False)
# silver_df.printSchema()

In [0]:
silver_df.printSchema()

In [0]:
%sql
select * from muse.bronze.job_muse_bronze_landing

In [0]:
%sql 
create schema muse.silver 

In [0]:

%sql
describe cata

In [0]:
dbutils.fs.rm("abfss://silver@musedatapipeline.dfs.core.windows.net/bronze/jobs", recurse=True)

In [0]:
silver_df.write \
    .format("delta") \
    .mode("append") \
    .option("path", "abfss://silver@musedatapipeline.dfs.core.windows.net/bronze/jobs") \
    .saveAsTable("muse.silver.job_muse_silver_staging")

In [0]:
%sql
drop table if exists muse.bronze.job_muse_silver

In [0]:
%sql

select * from muse.silver.job_muse_silver_staging;

In [0]:
%sql
select * from muse.silver.job_muse_silver_staging
where name like "%*%"

In [0]:
display(_sqldf)

# transformations